First version. Only valid ISBNs obtained by removing invalid characters (but result not checked for actual existence). Only non-zero ratings. Only users and books that have at least 3 ratings. 

In [1]:
%load_ext autoreload
%autoreload 2

In [64]:
import gc
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

from load_data import load_data
from valid_test_select import valid_test_select, valid_test_select_per_user
from initialize_model import initialize_mu_b_c
from helpers import *
from fit_model import (
    fit_model_no_UV,
    fit_model_full,
    fit_model_full_beta,
    fit_model_UV_only,
    infer_user_vector,
)

In [3]:
np.set_printoptions(linewidth=150, precision=6)  # 75, 8

# Loading Data

In [4]:
df = load_data()

In [5]:
df.agg(["min", "max", "nunique", "dtype", "count", "size"])

,userid,isbn,rating
min,2,0000001481,0
max,278854,9998150752,10
nunique,103371,328465,11
dtype,int32,object,int8
count,1135338,1135338,1135338
size,1135338,1135338,1135338


In [6]:
df = df[df.rating > 0]

In [7]:
df.agg(["min", "max", "nunique", "size"])

,userid,isbn,rating
min,8,000003827X,1
max,278854,9997555635,10
nunique,76331,179924,10
size,427261,427261,427261


# Creating Matrix Y

In [9]:
# Encoding userids and isbns as categories (integers 0 to N-1)

user_cats = df.userid.astype("category")
book_cats = df.isbn.astype("category")

In [10]:
# Creating sparse matrix and converting it to pd.DataFrame

Y_sparse = coo_matrix((df.rating.values, (user_cats.cat.codes, book_cats.cat.codes)))
Y = pd.DataFrame(Y_sparse.toarray())

In [11]:
# print("Y shape:", Y.shape)  # (76331, 179924)
# print("total entries:", (Y > 0).sum().sum())  # 427261
# print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))  # 5.6
# print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))  # 2.37

In [12]:
# Filtering out improbably active readers

# np.sort(np.partition((Y > 0).sum(axis=1), -20)[-20:])
Y = Y.loc[(Y > 0).sum(axis=1) < 2000, :]

In [13]:
# Filtering users and books with enough observations

min_rats = 5
old_rows, old_cols = Y.shape
while True:
    Y_pos = Y > 0  # type: ignore
    user_rats = Y_pos.sum(axis=1)  # type: ignore
    book_rats = Y_pos.sum(axis=0)  # type: ignore
    new_rows = (user_rats >= min_rats).sum()
    new_cols = (book_rats >= min_rats).sum()
    # print(new_rows, new_cols)
    Y = Y.loc[user_rats >= min_rats, book_rats >= min_rats]  # type: ignore
    if (old_rows == new_rows) and (old_cols == new_cols):
        break
    old_rows, old_cols = new_rows, new_cols

del Y_sparse, Y_pos, user_rats, book_rats, new_rows, new_cols, old_rows, old_cols
gc.collect()

40

In [14]:
print("Y shape:", Y.shape)  # (6916, 8903) |3: (14903, 22661)
print("total entries:", (Y > 0).sum().sum())  # 113019 |3: 182612
print("avg # ratings per user:", f"{(Y > 0).sum(axis=1).mean():.2f}")  # 16.34 |3: 12.25
print("median # ratings per user:", f"{(Y > 0).sum(axis=1).median():.2f}")  # 9.00
print("avg # ratings per book:", f"{(Y > 0).sum(axis=0).mean():.2f}")  # 12.69 |3: 8.06
print("median # ratings per book:", f"{(Y > 0).sum(axis=0).median():.2f}")  # 8.00

Y shape: (6916, 8903)
total entries: 113019
avg # ratings per user: 16.34
median # ratings per user: 9.00
avg # ratings per book: 12.69
median # ratings per book: 8.00


In [15]:
# Converting Y to numpy array

Y_columns = Y.columns.copy()
Y = Y.to_numpy()

# Final training

In [17]:
# # Selecting validation and test sets
# Y_train, valid_data, _ = valid_test_select(Y, 30_000, 0, seed=123)
Y_train, valid_data, _ = valid_test_select_per_user(Y, include_test=False, seed=123)
Y_csr = csr_matrix(Y_train)
Y_csc = Y_csr.tocsc()

In [18]:
mus, bs, cs, Us, Vs, inters, Y_preds = {}, {}, {}, {}, {}, {}, {}
mu_init, b_init, c_init = initialize_mu_b_c(Y_train)

In [95]:
test_rmses = {
    "bias-only": 1.584585,
    "full_no_beta": 1.583954,
    "full_w_beta": 1.767088,
    "UV-only": 6.038851,
    "random": 8.028258,
}
test_ndcgs = {
    "bias-only": 0.000562,
    "full_no_beta": 0.001182,
    "full_w_beta": 0.012716,
    "UV-only": 0.0255,
}
valid_ndcgs = {
    "bias-only": 0.000541,
    "full_no_beta": 0.001168,
    "full_w_beta": 0.022164,
    "UV-only": 0.040165,
    "random": 0.000201,
}

## Bias-only model

In [19]:
print("######## bias-only:", end="     ")
mu, b, c = fit_model_no_UV(
    Y_train, Y_csr, Y_csc, mu_init, b_init, c_init, 3.7, valid_data, 1e-5, info=1
)
mus["bias-only"] = mu
bs["bias-only"] = b
cs["bias-only"] = c
Y_preds["bias-only"] = mu + b[:, None] + c[None, :]

######## bias-only:     results: counter =  17, max_norm_diff =    0.09, valid_rmse = 1.568282, valid_ndcg = 0.000541, valid_rmse_diff = 9.3e-06, valid_ndcg_diff = 0      


## Full model without beta

In [20]:
k, l_b, l_f = 384, 4, 9.5

In [21]:
print(f"######## {k = :>3}:", end="     ")
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 0.5, size=(Y.shape[0], k))
V = rng.normal(0, 0.5, size=(Y.shape[1], k))
mu, b, c, U, V = fit_model_full(
    Y_train,
    Y_csr,
    Y_csc,
    mu_init,
    b_init,
    c_init,
    U,
    V,
    l_b,
    l_f,
    valid_data,
    1e-5,
    info=1,
)
mus["full_no_beta"] = mu
bs["full_no_beta"] = b
cs["full_no_beta"] = c
Us["full_no_beta"] = U
Vs["full_no_beta"] = V
inter = U @ V.T
inters["full_no_beta"] = inter
Y_preds["full_no_beta"] = mu + b[:, None] + c[None, :] + inter

######## k = 384:     results: counter =  11, max_norm_diff =    1.95, valid_rmse = 1.564413, valid_ndcg = 0.001168


## Full model with beta

In [23]:
k, l_b, l_f, beta = 384, 10000, 11, 0.1

In [24]:
print(f"{k = :>3}, {l_b = :>5.2f}, {l_f = :>5.2f}, {beta = }", end=", ")
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 0.5, size=(Y.shape[0], k))
V = rng.normal(0, 0.5, size=(Y.shape[1], k))
mu, b, c, U, V = fit_model_full_beta(
    Y_train, Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, beta, valid_data, 1e-5, info=1
)

mus["full_w_beta"] = mu
bs["full_w_beta"] = b
cs["full_w_beta"] = c
Us["full_w_beta"] = U
Vs["full_w_beta"] = V
inter = U @ V.T
inters["full_w_beta"] = inter
Y_preds["full_w_beta"] = mu + b[:, None] + beta * c[None, :] + inter

k = 384, l_b = 10000.00, l_f = 11.00, beta = 0.1, results: counter =   2, max_norm_diff =   32.12, valid_rmse = 1.760840, valid_ndcg = 0.022164


## UV-only model

In [25]:
k, l_f = 384, 21

In [26]:
print(f"{k = :<3}, {l_f = :<4}", end=", ")
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 0.5, size=(Y.shape[0], k))
V = rng.normal(0, 0.5, size=(Y.shape[1], k))
U, V = fit_model_UV_only(Y_train, Y_csr, Y_csc, U, V, l_f, valid_data, 1e-5, info=1)
Us["UV-only"] = U
Vs["UV-only"] = V
Y_preds["UV-only"] = U @ V.T

k = 384, l_f = 21  , results: counter =   2, max_norm_diff =  123.44, valid_rmse = 5.800090, valid_ndcg = 0.040165


# Creating Recommendations

In [28]:
rs = {}

# taking only predictions made for originally unseen entries
mask = Y > 0
K = 5

In [29]:
popularity = mask.sum(axis=0)
log_pop = np.log1p(popularity)
log_pop_mean_uniform = log_pop.mean()
weights = popularity / popularity.sum()
log_pop_mean_data = (weights * log_pop).sum()

mean_ratings = np.mean(Y, axis=0, where=Y > 0)
global_mean_rating = Y[mask].mean()

In [31]:
rs["bias-only"] = recommend(Y_preds["bias-only"] * (~mask), cs["bias-only"], K)

In [38]:
rs["full_no_beta"] = recommend(Y_preds["full_no_beta"] * (~mask), cs["full_no_beta"], K)

In [39]:
rs["full_w_beta"] = recommend(Y_preds["full_w_beta"] * (~mask), cs["full_w_beta"], K)

In [40]:
rs["UV-only"] = recommend(Y_preds["UV-only"] * (~mask), mean_ratings, K)

In [ ]:
scores = Y_preds["UV-only"] - 0.1 * log_pop[None, :]
rs["UV-only.1"] = recommend(scores * (~mask), mean_ratings, K)
valid_ndcgs["UV-only.1"] = get_ndcg(scores, Y_train, valid_data, 5)

In [107]:
scores = Y_preds["UV-only"] - 0.2 * log_pop[None, :]
rs["UV-only.2"] = recommend(scores * (~mask), mean_ratings, K)
valid_ndcgs["UV-only.2"] = get_ndcg(scores, Y_train, valid_data, 5)

In [ ]:
scores = Y_preds["UV-only"] - 0.3 * log_pop[None, :]
rs["UV-only.3"] = recommend(scores * (~mask), mean_ratings, K)
valid_ndcgs["UV-only.3"] = get_ndcg(scores, Y_train, valid_data, 5)

In [ ]:
scores = Y_preds["UV-only"] - 0.4 * log_pop[None, :]
rs["UV-only.4"] = recommend(scores * (~mask), mean_ratings, K)
valid_ndcgs["UV-only.4"] = get_ndcg(scores, Y_train, valid_data, 5)

In [ ]:
scores = Y_preds["UV-only"] - 0.5 * log_pop[None, :]
rs["UV-only.5"] = recommend(scores * (~mask), mean_ratings, K)
valid_ndcgs["UV-only.5"] = get_ndcg(scores, Y_train, valid_data, 5)

In [50]:
scores = np.random.normal(size=Y.shape)
rs["random"] = recommend(scores * (~mask), mean_ratings, K)
print(get_ndcg(scores, Y_train, valid_data, 5))
print(get_rmse(np.random.normal(size=len(valid_data)), valid_data))

0.0005823118543706949
8.028258462399748


# Computing metrics

In [108]:
models = [
    "random",
    "bias-only",
    "full_no_beta",
    "full_w_beta",
    "UV-only",
    "UV-only.1",
    "UV-only.2",
    "UV-only.3",
    "UV-only.4",
    "UV-only.5",
]
for key in models:
    r = rs[key]
    print(
        key.ljust(12) + ":",
        "NDCG:",
        f"{valid_ndcgs[key]:.6f}",
        "| RMSE:",
        test_rmses.get(key, "--------"),
        "| AWB (share, 0-1):",
        f"{agreement_with_bias_only(r, rs['bias-only']):.3f}",
        "| MOV (share, 0-1):",
        f"{mean_overlap(r):.4f}",
        "| COV (share, 0-1):",
        f"{coverage(r, Y.shape[1]):.4f}",
        "| ARP (log, <0>):",
        f"{rel_log_arp(r, log_pop, log_pop_mean_data):6.3f}",
        "| RMR: (share, <1>)",
        f"{rel_mean_rating(r, mean_ratings, global_mean_rating):.3f}",
        "| ENT: (share, 0-1)",
        f"{entropy(r, Y.shape[1]):.5f}",
    )

random      : NDCG: 0.000201 | RMSE: 8.028258 | AWB (share, 0-1): 0.001 | MOV (share, 0-1): 0.0006 | COV (share, 0-1): 0.9809 | ARP (log, <0>): -0.656 | RMR: (share, <1>) 0.993 | ENT: (share, 0-1) 0.98514
bias-only   : NDCG: 0.000541 | RMSE: 1.584585 | AWB (share, 0-1): 1.000 | MOV (share, 0-1): 0.9945 | COV (share, 0-1): 0.0009 | ARP (log, <0>): -0.031 | RMR: (share, <1>) 1.231 | ENT: (share, 0-1) 0.17868
full_no_beta: NDCG: 0.001168 | RMSE: 1.583954 | AWB (share, 0-1): 0.755 | MOV (share, 0-1): 0.8013 | COV (share, 0-1): 0.0138 | ARP (log, <0>):  0.045 | RMR: (share, <1>) 1.227 | ENT: (share, 0-1) 0.22981
full_w_beta : NDCG: 0.022164 | RMSE: 1.767088 | AWB (share, 0-1): 0.001 | MOV (share, 0-1): 0.0351 | COV (share, 0-1): 0.1578 | ARP (log, <0>):  1.261 | RMR: (share, <1>) 0.984 | ENT: (share, 0-1) 0.63972
UV-only     : NDCG: 0.040165 | RMSE: 6.038851 | AWB (share, 0-1): 0.000 | MOV (share, 0-1): 0.0932 | COV (share, 0-1): 0.1068 | ARP (log, <0>):  1.816 | RMR: (share, <1>) 1.060 | E

## Harry Potter predictions

In [ ]:
#     0747561966,Harry Potter and the Philosopher's Stone,J.K. Rowling,2003,Bloomsbury
#     059035342X,Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)),J. K. Rowling,1999,Arthur A. Levine Books
# ### 0590353403,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,1998,Scholastic
#     043936213X,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,2001,Scholastic
#     043920352X,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,2000,Scholastic
#     0439064872,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,2000,Scholastic
#     0439420105,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,2002,Scholastic
# ### 0439064864,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,1999,Scholastic
#     0747545774,Harry Potter and the Chamber of Secrets,J. K. Rowling,2000,Trafalgar Square (J)
#     1551922444,Harry Potter and the Chamber of Secrets,J. K. Rowling,2000,Raincoast Book Distribution
#     0439554896,Harry Potter and the Chamber of Secrets (Harry Potter),J. K. Rowling,2003,Arthur A. Levine Books
# ### 0439136350,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,1999,Scholastic
#     0439136369,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,2001,Scholastic
#     043965548X,Harry Potter and the Prisoner of Azkaban (Harry Potter),J.K. Rowling,2004,Scholastic Paperbacks
#     0747545111,Harry Potter and the Prisoner of Azkaban,J. K. Rowling,2000,Bloomsbury Pub Ltd
#     0786222743,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,2000,Thorndike Press
# ### 0439139597,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,2000,Scholastic
#     0439139600,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,2002,Scholastic Paperbacks
#     0613496744,Harry Potter and the Goblet of Fire,J. K. Rowling,2002,Sagebrush Bound
#     043955490X,Harry Potter and the Goblet of Fire (Harry Potter),J. K. Rowling,2003,Arthur A. Levine Books
#     043935806X,Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,2003,Scholastic
#     0439567610,Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,2003,Scholastic
#     0439358078,Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,2004,Scholastic Paperbacks
#     043935806x,Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,2003,Scholastic

In [117]:
HP = ["0590353403", "0439064864", "0439136350", "0439139597"]

In [118]:
new_item_ids = [get_Ypos(isbn, book_cats, Y_columns) for isbn in HP]
new_ratings = [10, 10, 10, 10]

In [119]:
new_u = infer_user_vector(Vs["UV-only"], new_item_ids, new_ratings, lambda_fact=21)

In [120]:
new_preds = new_u @ V.T - 0.3 * log_pop
new_preds[new_item_ids] = 0

In [126]:
for pos in recommend(new_preds[None, :], mean_ratings, 5).flatten():
    isbn = get_isbn(pos, book_cats, Y_columns)
    print(isbn, end=", ")
    print(get_book_info(isbn))

043935806X, Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,2003,Scholastic
059035342X, Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)),J. K. Rowling,1999,Arthur A. Levine Books
0439139600, Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,2002,Scholastic Paperbacks
0439064872, Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,2000,Scholastic
0439136369, Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,2001,Scholastic
